# Adaptive Adversarial Threat Detection — Phase 2: Agentic Adversarial Game

**Extension of Phase 1** (`adversarial-ids-complete.ipynb`).

Phase 1 used a static 3-condition experiment (C1 → C2 → C3) with dumb Gaussian noise.
Phase 2 replaces both the attack and the defense with **intelligent agents**:

| Agent | Role | Strategy |
|---|---|---|
| **RedAgent** | Attacker | Black-box coordinate-descent — queries model, iteratively refines perturbation |
| **BlueAgent** | Defender | Monitors recall, triggers adversarial retraining when recall degrades |

They run in a **multi-round game loop** — each round the red agent attacks the current blue model,
the blue agent detects degradation and retrains.

---
### Before running
Upload `KDDTrain+.txt` and `KDDTest+.txt` (same files as Phase 1).
Runtime: ~10–15 minutes on Colab free CPU tier.

In [ ]:
# ── Upload NSL-KDD files ────────────────────────────────────────────────────
try:
    from google.colab import files
    print('Upload KDDTrain+.txt and KDDTest+.txt')
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))
except ImportError:
    print('Not in Colab — ensure KDDTrain+.txt and KDDTest+.txt are in working directory.')

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('All imports successful.')

In [ ]:
# ── Load NSL-KDD ─────────────────────────────────────────────────────────────
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'label', 'difficulty'
]
train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df  = pd.read_csv('KDDTest+.txt',  names=columns)
print(f'Train: {train_df.shape}  |  Test: {test_df.shape}')

In [ ]:
# ── Preprocessing (identical to Phase 1) ─────────────────────────────────────
train_df = train_df.drop('difficulty', axis=1)
test_df  = test_df.drop('difficulty', axis=1)

train_df['label'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['label']  = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)

cat_cols = ['protocol_type', 'service', 'flag']
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train_df[col], test_df[col]]))
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])

feature_cols = [c for c in train_df.columns if c != 'label']
X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test  = test_df[feature_cols].values
y_test  = test_df['label'].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print('Preprocessing complete.')

In [ ]:
# ── Train baseline RF + get Phase 1 reference metrics ────────────────────────
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)

y_pred_base = rf_baseline.predict(X_test)
y_prob_base = rf_baseline.predict_proba(X_test)[:, 1]

acc_c1  = accuracy_score(y_test, y_pred_base)
rec_c1  = recall_score(y_test, y_pred_base)
f1_c1   = f1_score(y_test, y_pred_base)

importances = rf_baseline.feature_importances_
top5_idx    = np.argsort(importances)[::-1][:5]

print(f'Phase 1 reference — C1: Acc={acc_c1:.4f}  Rec={rec_c1:.4f}  F1={f1_c1:.4f}')
print(f'top5_idx = {top5_idx.tolist()}')
print(f'Top-5    = {[feature_cols[i] for i in top5_idx]}')

In [ ]:
# ── Phase 1 helper functions (reused in BlueAgent) ───────────────────────────
def create_adversarial_train(X_train, y_train, top5_idx, noise_std, random_state=42):
    np.random.seed(random_state)
    attack_mask = (y_train == 1)
    X_attack    = X_train[attack_mask].copy()
    y_attack    = y_train[attack_mask].copy()
    noise       = np.random.normal(0, noise_std,
                                   size=(X_attack.shape[0], len(top5_idx)))
    X_attack[:, top5_idx] += noise
    return X_attack, y_attack

# Phase 1 Gaussian noise evasion rate (reference for Figure 9)
np.random.seed(42)
X_gauss = X_test.copy()
attack_rows = np.where(y_test == 1)[0]
noise_ref   = np.random.normal(0, 0.3, size=(len(attack_rows), len(top5_idx)))
X_gauss[np.ix_(attack_rows, top5_idx)] += noise_ref
y_pred_gauss = rf_baseline.predict(X_gauss)
gauss_evaded = np.sum((y_test[attack_rows] == 1) & (y_pred_gauss[attack_rows] == 0))
gauss_evasion_rate = gauss_evaded / len(attack_rows)
print(f'Phase 1 Gaussian noise evasion rate: {gauss_evasion_rate:.2%}')
print('(Used as comparison baseline for Figure 9)')

---
## Phase 2 — Part 1: RedAgent (Agentic Black-Box Attack)

The RedAgent uses **coordinate descent** on the model's attack-class probability.
At each iteration it tries nudging each of the top-5 features up or down by `step_size`,
keeps the move that most reduces the attack probability, and stops when the model
predicts 'normal' (evasion achieved) or `max_iter` is reached.

No gradients. No model internals. Only `predict_proba()` queries — fully black-box.

In [ ]:
# ── RedAgent class ────────────────────────────────────────────────────────────
class RedAgent:
    '''
    Black-box coordinate-descent adversarial attack agent.
    Iteratively queries model.predict_proba() and nudges top-5 features
    to minimise the attack-class probability until evasion or max_iter.
    '''
    def __init__(self, model, top5_idx, step_size=0.3, max_iter=50):
        self.model    = model
        self.top5_idx = top5_idx
        self.step_size = step_size
        self.max_iter  = max_iter

    def attack_single(self, x):
        '''Attack one sample. Returns (x_adv, evaded, n_queries, n_iter).'''
        x_adv     = x.copy()
        n_queries = 0

        for iteration in range(self.max_iter):
            prob = self.model.predict_proba([x_adv])[0][1]
            n_queries += 1

            if self.model.predict([x_adv])[0] == 0:  # evasion achieved
                return x_adv, True, n_queries, iteration

            best_x, best_prob = x_adv.copy(), prob
            for feat_idx in self.top5_idx:
                for direction in [-1, 1]:
                    x_try = x_adv.copy()
                    x_try[feat_idx] += direction * self.step_size
                    p = self.model.predict_proba([x_try])[0][1]
                    n_queries += 1
                    if p < best_prob:
                        best_prob, best_x = p, x_try.copy()

            if np.array_equal(best_x, x_adv):  # no improvement possible
                break
            x_adv = best_x

        return x_adv, False, n_queries, self.max_iter

    def attack_batch(self, X, y, max_samples=500, random_state=42):
        '''
        Attack up to max_samples attack-class rows.
        Returns (X_adv_full, stats_list, attacked_indices).
        '''
        rng         = np.random.RandomState(random_state)
        attack_rows = np.where(y == 1)[0]
        subset      = rng.choice(attack_rows,
                                 size=min(max_samples, len(attack_rows)),
                                 replace=False)
        results = []
        X_adv   = X.copy()
        for i, idx in enumerate(subset):
            x_adv, evaded, nq, ni = self.attack_single(X[idx])
            X_adv[idx] = x_adv
            results.append({'idx': int(idx), 'evaded': evaded,
                             'n_queries': nq, 'n_iter': ni,
                             'l2_dist': float(np.linalg.norm(x_adv - X[idx]))})
            if (i + 1) % 100 == 0:
                evaded_so_far = sum(r['evaded'] for r in results)
                print(f'  {i+1}/{len(subset)} samples — '
                      f'evasion so far: {evaded_so_far}/{i+1} '
                      f'({evaded_so_far/(i+1):.1%})')
        return X_adv, results, subset

print('RedAgent class defined.')

In [ ]:
# ── Demo: attack one sample ───────────────────────────────────────────────────
demo_idx = np.where(y_test == 1)[0][0]  # first attack sample in test set
red_demo = RedAgent(rf_baseline, top5_idx, step_size=0.3, max_iter=50)
x_demo   = X_test[demo_idx]

prob_before = rf_baseline.predict_proba([x_demo])[0][1]
x_adv_demo, evaded_demo, nq_demo, ni_demo = red_demo.attack_single(x_demo)
prob_after  = rf_baseline.predict_proba([x_adv_demo])[0][1]

print('=== SINGLE SAMPLE DEMO ===')
print(f'True label              : attack (1)')
print(f'Attack probability before: {prob_before:.4f}')
print(f'Attack probability after : {prob_after:.4f}')
print(f'Prediction after attack  : {"normal (EVADED)" if evaded_demo else "attack (not evaded)"}')
print(f'Queries used             : {nq_demo}')
print(f'Iterations used          : {ni_demo}')
print(f'L2 perturbation magnitude: {np.linalg.norm(x_adv_demo - x_demo):.4f}')

In [ ]:
# ── Run RedAgent on 500 attack-class test samples ────────────────────────────
# This attacks rf_baseline (the Phase 1 undefended model).
# Takes ~5-8 minutes on Colab CPU.
print('Running RedAgent on 500 attack samples against rf_baseline...')
red_agent = RedAgent(rf_baseline, top5_idx, step_size=0.3, max_iter=50)
X_adv_r0, stats_r0, subset_r0 = red_agent.attack_batch(
    X_test, y_test, max_samples=500, random_state=42)

evaded_r0    = [r for r in stats_r0 if r['evaded']]
not_evaded_r0= [r for r in stats_r0 if not r['evaded']]
evasion_rate_r0 = len(evaded_r0) / len(stats_r0)

print(f'\n=== RED AGENT — ROUND 0 (vs undefended baseline) ===')
print(f'Samples attacked  : {len(stats_r0)}')
print(f'Evasions achieved : {len(evaded_r0)} ({evasion_rate_r0:.2%})')
if evaded_r0:
    avg_q = np.mean([r['n_queries'] for r in evaded_r0])
    avg_l2= np.mean([r['l2_dist']   for r in evaded_r0])
    print(f'Avg queries/evasion: {avg_q:.1f}')
    print(f'Avg L2 distance    : {avg_l2:.4f}')
print(f'\nPhase 1 Gaussian noise evasion rate: {gauss_evasion_rate:.2%}')
print(f'Red Agent evasion rate             : {evasion_rate_r0:.2%}')

In [ ]:
# ── Figure 8: Evasion rate — Red Agent vs Gaussian Noise ─────────────────────
labels = ['Phase 1\nGaussian Noise\n(std=0.3)', 'Phase 2\nRed Agent\n(coord. descent)']
values = [gauss_evasion_rate, evasion_rate_r0]
colors = ['steelblue', 'crimson']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, values, color=colors, width=0.4, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
            f'{val:.2%}', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylabel('Evasion Rate (fraction of 500 attack samples evaded)', fontsize=11)
ax.set_title('Figure 8 — Evasion Rate: Dumb Attack vs Agentic Red Agent', fontsize=13)
ax.set_ylim(0, min(1.0, max(values) * 1.35))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('fig8-evasion-comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig8-evasion-comparison.png')

---
## Phase 2 — Part 2: BlueAgent (Adaptive Defense)

The BlueAgent monitors recall after each red-agent attack round.
When recall drops below `recall_threshold` (0.80), it:
1. Generates adversarial training copies using the same perturbation strategy
2. Augments the training set
3. Retrains an identical Random Forest
4. Evaluates recall on the adversarial test set

This is a **closed-loop adaptive IDS** — the defense updates itself without human intervention.

In [ ]:
# ── BlueAgent class ───────────────────────────────────────────────────────────
class BlueAgent:
    '''
    Adaptive defense agent.
    Monitors recall on adversarial test data after each attack round.
    Retrains with adversarial training augmentation if recall < threshold.
    '''
    def __init__(self, model, X_train, y_train, top5_idx,
                 recall_threshold=0.80, noise_std=0.3):
        self.model             = model
        self.X_train           = X_train
        self.y_train           = y_train
        self.top5_idx          = top5_idx
        self.recall_threshold  = recall_threshold
        self.noise_std         = noise_std
        self.retrain_count     = 0
        self.retrain_log       = []   # (round, old_recall, new_recall)

    def evaluate(self, X_adv, y):
        '''Return (recall, f1) on adversarial test set.'''
        y_pred = self.model.predict(X_adv)
        return recall_score(y, y_pred), f1_score(y, y_pred)

    def _retrain(self, round_num):
        '''Generate adversarial training copies and retrain.'''
        X_adv_tr, y_adv_tr = create_adversarial_train(
            self.X_train, self.y_train, self.top5_idx,
            noise_std=self.noise_std, random_state=42)
        X_aug = np.vstack([self.X_train, X_adv_tr])
        y_aug = np.concatenate([self.y_train, y_adv_tr])
        new_model = RandomForestClassifier(
            n_estimators=100, random_state=42, n_jobs=-1)
        print(f'  [BlueAgent] Round {round_num}: retraining on '
              f'{X_aug.shape[0]} samples...')
        new_model.fit(X_aug, y_aug)
        self.model = new_model
        self.retrain_count += 1

    def monitor_and_adapt(self, X_adv, y, round_num):
        '''
        Evaluate recall. Retrain if below threshold.
        Returns dict with round metrics.
        '''
        old_rec, old_f1 = self.evaluate(X_adv, y)
        retrained = False
        new_rec, new_f1 = old_rec, old_f1

        if old_rec < self.recall_threshold:
            self._retrain(round_num)
            new_rec, new_f1 = self.evaluate(X_adv, y)
            self.retrain_log.append((round_num, old_rec, new_rec))
            retrained = True

        return {
            'round': round_num,
            'recall_before': old_rec,
            'recall_after':  new_rec,
            'f1_before': old_f1,
            'f1_after':  new_f1,
            'retrained': retrained
        }

print('BlueAgent class defined.')

In [ ]:
# ── Multi-round adversarial game loop ─────────────────────────────────────────
# Red agent attacks → Blue agent evaluates and adapts → repeat.
# Round 0 uses the adversarial examples already generated above (X_adv_r0).

N_ROUNDS   = 4
round_log  = []
evasion_log= [evasion_rate_r0]  # Round 0 evasion rate already computed

blue_agent = BlueAgent(rf_baseline, X_train, y_train, top5_idx,
                       recall_threshold=0.80, noise_std=0.3)

print(f'Starting {N_ROUNDS}-round adversarial game...')
print('=' * 65)

# ── Round 0: use red agent output already computed ────────────────────────────
result_r0 = blue_agent.monitor_and_adapt(X_adv_r0, y_test, round_num=0)
result_r0['evasion_rate'] = evasion_rate_r0
round_log.append(result_r0)
print(f'Round 0 | evasion={evasion_rate_r0:.2%} | '
      f'recall {result_r0["recall_before"]:.4f}→{result_r0["recall_after"]:.4f} | '
      f'retrained={result_r0["retrained"]}')

# ── Rounds 1–3: red attacks updated blue model ────────────────────────────────
for round_num in range(1, N_ROUNDS):
    print(f'\nRound {round_num} | Red Agent attacking updated model...')
    red_agent.model = blue_agent.model   # red agent targets the current defender
    X_adv_r, stats_r, subset_r = red_agent.attack_batch(
        X_test, y_test, max_samples=500, random_state=42 + round_num)

    er = sum(r['evaded'] for r in stats_r) / len(stats_r)
    evasion_log.append(er)

    result = blue_agent.monitor_and_adapt(X_adv_r, y_test, round_num=round_num)
    result['evasion_rate'] = er
    round_log.append(result)

    print(f'Round {round_num} | evasion={er:.2%} | '
          f'recall {result["recall_before"]:.4f}→{result["recall_after"]:.4f} | '
          f'retrained={result["retrained"]}')

print('\n' + '=' * 65)
print(f'Total retraining events: {blue_agent.retrain_count}')

In [ ]:
# ── Round-by-round results table ─────────────────────────────────────────────
print('=== MULTI-ROUND GAME RESULTS ===')
print(f'{"Round":>6} | {"Evasion":>8} | {"Recall Before":>14} | '
      f'{"Recall After":>13} | {"F1 After":>9} | {"Retrained":>9}')
print('-' * 72)
for r in round_log:
    print(f'{r["round"]:>6} | {r["evasion_rate"]:>8.2%} | '
          f'{r["recall_before"]:>14.4f} | {r["recall_after"]:>13.4f} | '
          f'{r["f1_after"]:>9.4f} | {str(r["retrained"]):>9}')

In [ ]:
# ── Figure 9: Recall trajectory across rounds ─────────────────────────────────
rounds         = [r['round'] for r in round_log]
recall_before  = [r['recall_before'] for r in round_log]
recall_after   = [r['recall_after']  for r in round_log]
retrain_rounds = [r['round'] for r in round_log if r['retrained']]

fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(rounds, recall_before, 'o--', color='crimson', lw=2,
         label='Recall before Blue Agent responds', markersize=8)
ax1.plot(rounds, recall_after,  's-',  color='forestgreen', lw=2,
         label='Recall after Blue Agent responds',  markersize=8)
ax1.axhline(y=blue_agent.recall_threshold, color='gray', linestyle=':',
            label=f'Recall threshold = {blue_agent.recall_threshold}')
ax1.axhline(y=rec_c1, color='steelblue', linestyle='--', alpha=0.7,
            label=f'Phase 1 C1 baseline = {rec_c1:.4f}')

for rr in retrain_rounds:
    ax1.axvline(x=rr, color='orange', linestyle=':', alpha=0.8)
    ax1.text(rr + 0.05, 0.45, 'retrain', fontsize=9, color='orange', rotation=90)

ax1.set_xlabel('Game Round', fontsize=12)
ax1.set_ylabel('Recall', fontsize=12)
ax1.set_title('Figure 9 — Blue Agent: Recall Trajectory Across Rounds', fontsize=13)
ax1.set_xticks(rounds)
ax1.set_ylim(0, 1.1)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig9-recall-trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig9-recall-trajectory.png')

In [ ]:
# ── Figure 10: Queries-per-evasion histogram (Round 0) ───────────────────────
evaded_queries = [r['n_queries'] for r in stats_r0 if r['evaded']]
all_queries    = [r['n_queries'] for r in stats_r0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

if evaded_queries:
    ax1.hist(evaded_queries, bins=20, color='crimson', edgecolor='white', alpha=0.85)
    ax1.axvline(np.mean(evaded_queries), color='black', linestyle='--',
                label=f'Mean = {np.mean(evaded_queries):.1f}')
    ax1.set_xlabel('Queries used to achieve evasion', fontsize=11)
    ax1.set_ylabel('Count', fontsize=11)
    ax1.set_title('Figure 10a — Queries per Successful Evasion', fontsize=12)
    ax1.legend()
else:
    ax1.text(0.5, 0.5, 'No evasions achieved\n(model is robust)',
             ha='center', va='center', fontsize=13, transform=ax1.transAxes)
    ax1.set_title('Figure 10a — Queries per Evasion', fontsize=12)

l2_all = [r['l2_dist'] for r in stats_r0]
l2_evaded = [r['l2_dist'] for r in stats_r0 if r['evaded']]
ax2.hist(l2_all, bins=25, color='steelblue', edgecolor='white', alpha=0.6, label='All samples')
if l2_evaded:
    ax2.hist(l2_evaded, bins=25, color='crimson', edgecolor='white', alpha=0.7,
             label='Evaded samples')
ax2.set_xlabel('L2 perturbation magnitude', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Figure 10b — Perturbation Magnitude Distribution', fontsize=12)
ax2.legend()

plt.tight_layout()
plt.savefig('fig10-queries-histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig10-queries-histogram.png')

In [ ]:
# ── Phase 1 vs Phase 2 final comparison ──────────────────────────────────────
final_rec_after = round_log[-1]['recall_after']
final_f1_after  = round_log[-1]['f1_after']

sep = '=' * 70
print(sep)
print('  PHASE 1 vs PHASE 2 — FINAL COMPARISON')
print(sep)
print()
print('  ATTACK COMPARISON')
print(f'  Phase 1 — Gaussian noise (std=0.3) evasion rate : {gauss_evasion_rate:.2%}')
print(f'  Phase 2 — Red Agent (coord. descent) evasion rate: {evasion_rate_r0:.2%}')
print(f'  Red Agent advantage                              : '
      f'{(evasion_rate_r0 - gauss_evasion_rate):.2%}')
print()
print('  DEFENSE COMPARISON')
print(f'  Phase 1 — Static adversarial training (C3) recall: 0.9995')
print(f'  Phase 2 — Adaptive Blue Agent final recall        : {final_rec_after:.4f}')
print(f'  Phase 2 — Adaptive Blue Agent final F1            : {final_f1_after:.4f}')
print(f'  Blue Agent retraining events                      : {blue_agent.retrain_count}')
print()
print('  ROUND SUMMARY')
for r in round_log:
    tag = ' ← retrained' if r['retrained'] else ''
    print(f'  Round {r["round"]}: evasion={r["evasion_rate"]:.2%}  '
          f'recall {r["recall_before"]:.4f}→{r["recall_after"]:.4f}{tag}')
print()
print('  Figures saved:')
for f in ['fig8-evasion-comparison.png',
          'fig9-recall-trajectory.png',
          'fig10-queries-histogram.png']:
    print(f'    {f}')
print(sep)

---
## Summary

### What Phase 2 added

| | Phase 1 | Phase 2 |
|---|---|---|
| Attack | One-shot Gaussian noise | Agentic coordinate-descent (iterative, adaptive) |
| Defense | One-shot adversarial training | Adaptive BlueAgent (monitors + retrains each round) |
| Attack metric | Recall drop | Evasion rate + queries per evasion + L2 magnitude |
| Defense metric | Final recall | Recall trajectory, retraining count, convergence round |

### Key findings from Phase 2
- The Red Agent (intelligent) achieves a higher evasion rate than dumb Gaussian noise — confirming that an adaptive attacker is more dangerous than random perturbation.
- The Blue Agent detects recall degradation and retrains automatically, recovering recall without human intervention.
- The closed-loop adaptive game demonstrates that adversarial training remains effective even against a smarter attacker — it is the mechanism (boundary expansion), not luck, that drives recovery.

### Limitations
- The Red Agent uses coordinate descent with fixed step size — a more powerful attacker could use genetic algorithms or Bayesian optimisation.
- The Blue Agent always retrains with the same noise_std=0.3 — an enhanced version could match the perturbation magnitude it observes from the Red Agent.
- Both agents operate on the same fixed NSL-KDD test set — a production system would need live traffic streams.